# Policy-gradient corrections in miles, in one notebook

Each method changes the loss of a sampled token. They fix three different problems.

**Table 1. The three problems and the methods that fix them.**

| Problem | Ratio it looks at | Methods |
|---|---|---|
| 1. The inference engine and the trainer give the same token different probabilities (same weights, different numerics) | `rho = p_old / p_engine` | TIS, MIS, IcePop |
| 2. The weights moved since the tokens were sampled (several optimizer steps per rollout, or async) | `r = p_now / p_old` | PPO clip, OPSM, GSPO |
| 3. Drift: sampling from the engine adds a pull toward (or push away from) the engine's distribution | engine distribution `q` vs trainer `p` | score centering |

Decoupled PPO (miles `--use-tis`) and CISPO as miles implements it act on 1 and 2 together.

Names: TIS = truncated importance sampling, MIS = masked importance sampling, OPSM = off-policy sequence masking,
GSPO = group sequence policy optimization, CISPO = clipped importance-sampling-weight policy optimization.

Shapes: `B` batch (responses), `T` sequence length (tokens per response), `V` vocabulary size, `D` input size, `H` hidden units. Logits and probabilities are (B, T, V); the hidden layer is (B, T, H).

In [1]:
import torch
torch.set_printoptions(precision=2, sci_mode=False)
B, T, V, D, H = 2, 4, 3, 2, 4                          # batch of 2 responses, 4 tokens each, 3-token vocab, input size 2, 4 hidden
g = torch.Generator().manual_seed(0)
f64 = dict(dtype=torch.float64)

X = torch.randn(B, T, D, generator=g, **f64)           # (B, T, D)  input at each position


def logits(W1, W2):
    """W1: (D, H), W2: (H, V) -> z: (B, T, V). A two-layer network applied at every position."""
    h = torch.tanh(X @ W1)                             # (B, T, H)  hidden layer
    return h @ W2                                      # (B, T, V)  logits: one score per vocabulary token


W1_old = torch.randn(D, H, generator=g, **f64)         # (D, H)  weights at rollout time
W2_old = torch.randn(H, V, generator=g, **f64)         # (H, V)

# The engine samples the tokens with the same weights, but its numbers differ a bit (other kernels, other
# precision). Modeled as noise on the logits.
q = torch.softmax(logits(W1_old, W2_old) + 0.6 * torch.randn(B, T, V, generator=g, **f64), -1)   # (B, T, V)  engine probs
y = torch.multinomial(q.reshape(-1, V), 1, generator=g).reshape(B, T)                            # (B, T)     sampled tokens

# Several optimizer steps later the trainer's weights have moved.
W1 = (W1_old + 0.25 * torch.randn(D, H, generator=g, **f64)).requires_grad_()   # (D, H)
W2 = (W2_old + 0.25 * torch.randn(H, V, generator=g, **f64)).requires_grad_()   # (H, V)

A = torch.tensor([[+1.0], [-1.0]], **f64)              # (B, 1)  advantage: response 0 good, response 1 bad
N = B * T                                              # ()      number of tokens; every loss is a mean over them


e_y = torch.nn.functional.one_hot(y, V).double()       # (B, T, V)  one-hot of each sampled token


def pick(log_p):
    """log_p: (B, T, V) -> (B, T): the log-prob of each sampled token."""
    return (log_p * e_y).sum(-1)


with torch.no_grad():
    logp_old = pick(torch.log_softmax(logits(W1_old, W2_old), -1))    # (B, T)  trainer, rollout weights
    logp_eng = pick(torch.log(q))                                      # (B, T)  engine, rollout weights

rho = (logp_old - logp_eng).exp()                      # (B, T)  problem 1: trainer / engine, no gradient
print("rho = p_old / p_engine:\n", rho)

rho = p_old / p_engine:
 tensor([[1.43, 0.94, 0.75, 0.74],
        [1.38, 1.15, 0.29, 2.31]], dtype=torch.float64)


In [2]:
def per_token_weight(loss_fn):
    """loss_fn: logp_now (B, T) -> loss (). Returns w: (B, T).

    Every loss here except score centering reaches the logits only through logp_now, the log-prob of the
    sampled token. So the whole difference between methods is d loss / d logp_now. REINFORCE's is -A / N.
    w = (this loss's d loss / d logp_now) / (-A / N). w = 1: like REINFORCE. w = 0: the token is dropped.
    The gradient on the logits is then w * A * (p - e_y) / N, shape (B, T, V): each token's V logits get its w * A.
    """
    log_p = torch.log_softmax(logits(W1, W2), -1)      # (B, T, V)  current trainer, with gradient
    logp_now = pick(log_p)                             # (B, T)
    logp_now.retain_grad()
    loss_fn(logp_now).backward()                       # ()
    W1.grad, W2.grad = None, None
    return (logp_now.grad / (-A / N)).detach()         # (B, T)


def r_of(logp_now):
    """(B, T) -> (B, T). Problem 2: trainer now / trainer at rollout. Has gradient."""
    return (logp_now - logp_old).exp()


def ppo_terms(r, lo=0.8, hi=1.28):
    """r: (B, T) -> (B, T). PPO: -min(r*A, clamp(r)*A). Where the clamp wins, the term is a constant.
    The band [0.8, 1.28] is what our recipes set; miles's default upper bound is 1.2."""
    return -torch.min(r * A, torch.clamp(r, lo, hi) * A)

## Baselines: REINFORCE, PPO, CISPO

In [3]:
def reinforce(logp_now):
    return (-A * logp_now).mean()                                      # ()  no ratio

def ppo(logp_now):
    return ppo_terms(r_of(logp_now)).mean()                            # ()  w = r, or 0 where the clamp wins

def cispo_paper(logp_now):
    w = torch.clamp(r_of(logp_now), 0.0, 4.0).detach()                 # (B, T)  frozen: autograd skips it
    return (-w * A * logp_now).mean()                                  # ()  w = clamp(p_now / p_old, 0, 4)

def cispo_miles(logp_now):
    w = torch.clamp((logp_now - logp_eng).exp(), 0.0, 4.0).detach()    # (B, T)  divides by the ENGINE's prob
    return (-w * A * logp_now).mean()                                  # ()  miles tinker_losses.py:210-212 (sums; same up to scale)

with torch.no_grad():
    print("r = p_now / p_old:\n", r_of(pick(torch.log_softmax(logits(W1, W2), -1))))
for name, fn in [("REINFORCE", reinforce), ("PPO", ppo), ("CISPO (paper)", cispo_paper), ("CISPO (miles)", cispo_miles)]:
    print(f"{name} w:\n", per_token_weight(fn))
# PPO, response 0 (A > 0): it only clips r > 1.28; none here, so w = r.
# PPO, response 1 (A < 0): r < 0.8 is clipped -> w = 0. r = 2.95 is on the other side, so PPO leaves it uncapped.
# CISPO keeps every token. The miles version multiplies staleness (r) and mismatch (rho): p_now / p_engine = r * rho.

r = p_now / p_old:
 tensor([[1.05, 0.84, 0.38, 0.51],
        [0.77, 2.95, 0.21, 0.74]], dtype=torch.float64)
REINFORCE w:
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]], dtype=torch.float64)
PPO w:
 tensor([[1.05, 0.84, 0.38, 0.51],
        [0.00, 2.95, 0.00, 0.00]], dtype=torch.float64)
CISPO (paper) w:
 tensor([[1.05, 0.84, 0.38, 0.51],
        [0.77, 2.95, 0.21, 0.74]], dtype=torch.float64)
CISPO (miles) w:
 tensor([[1.51, 0.79, 0.29, 0.37],
        [1.07, 3.40, 0.06, 1.70]], dtype=torch.float64)


## Problem 1: engine vs trainer, per token (TIS, MIS, IcePop)

Each makes a frozen weight `c` from `rho` and multiplies PPO's term by it (miles `losses.py:234-265`). TIS with
PPO's clip on `r` is decoupled PPO: miles computes it when `--use-tis` is on and `--use-rollout-logprobs` is off.

In [4]:
c_tis    = torch.clamp(rho, 0.0, 2.0)                                               # (B, T)  TIS: cap at 2
c_mis    = torch.where((rho >= 0.5) & (rho <= 2.0), rho, torch.zeros_like(rho))     # (B, T)  MIS: drop outside [0.5, 2]
c_icepop = torch.where(rho <= 2.0, rho, torch.zeros_like(rho))                      # (B, T)  IcePop: drop above 2

def tis(logp_now):    return (c_tis    * ppo_terms(r_of(logp_now))).mean()           # ()
def mis(logp_now):    return (c_mis    * ppo_terms(r_of(logp_now))).mean()           # ()
def icepop(logp_now): return (c_icepop * ppo_terms(r_of(logp_now))).mean()           # ()

print("c, TIS   :\n", c_tis)       # response 1 token 4: rho 2.31 -> 2.00 (capped)
print("c, MIS   :\n", c_mis)       # response 1 tokens 3, 4: 0.29 and 2.31 -> 0 (dropped)
print("c, IcePop:\n", c_icepop)    # response 1 token 4: 2.31 -> 0 (dropped)
print("TIS w = c x PPO w:\n", per_token_weight(tis))
# On this batch MIS and IcePop give the same w as TIS: the tokens where their c differs
# (response 1, tokens 3 and 4) are tokens PPO already set to 0.

c, TIS   :
 tensor([[1.43, 0.94, 0.75, 0.74],
        [1.38, 1.15, 0.29, 2.00]], dtype=torch.float64)
c, MIS   :
 tensor([[1.43, 0.94, 0.75, 0.74],
        [1.38, 1.15, 0.00, 0.00]], dtype=torch.float64)
c, IcePop:
 tensor([[1.43, 0.94, 0.75, 0.74],
        [1.38, 1.15, 0.29, 0.00]], dtype=torch.float64)
TIS w = c x PPO w:
 tensor([[1.51, 0.79, 0.29, 0.37],
        [0.00, 3.40, 0.00, 0.00]], dtype=torch.float64)


## Problem 2: staleness across steps, per response (OPSM, GSPO)

In [5]:
def opsm(logp_now, delta=1e-4):
    # Drop a whole response if it lost (A < 0) and the policy already moved away from it.
    # miles compares against logp_old from the trainer; with --use-rollout-logprobs it uses the engine's instead.
    seq_kl = (logp_old - logp_now).mean(-1, keepdim=True).detach()        # (B, 1)
    keep = ~((A < 0) & (seq_kl > delta))                                   # (B, 1)
    return (keep * ppo_terms(r_of(logp_now))).mean()                      # ()

def gspo(logp_now, lo=0.8, hi=1.28):
    # One ratio per response: the geometric mean of its token ratios, clipped as a whole.
    s = (logp_now - logp_old).mean(-1, keepdim=True).exp()                # (B, 1)  gradient flows to every token
    return (-torch.min(s * A, torch.clamp(s, lo, hi) * A)).expand(B, T).mean()   # ()

with torch.no_grad():
    lp = pick(torch.log_softmax(logits(W1, W2), -1))                      # (B, T)
    print("per-response KL (OPSM)   :", (logp_old - lp).mean(-1))        # (B,)  response 1: 0.27 > 1e-4 with A < 0 -> dropped
    print("per-response ratio (GSPO):", (lp - logp_old).mean(-1).exp())  # (B,)  response 1: 0.77 < 0.8 with A < 0 -> clipped
print("OPSM w:\n", per_token_weight(opsm))
print("GSPO w (one value per response):\n", per_token_weight(gspo))

per-response KL (OPSM)   : tensor([0.44, 0.27], dtype=torch.float64)
per-response ratio (GSPO): tensor([0.64, 0.77], dtype=torch.float64)
OPSM w:
 tensor([[1.05, 0.84, 0.38, 0.51],
        [0.00, 0.00, 0.00, 0.00]], dtype=torch.float64)
GSPO w (one value per response):
 tensor([[0.64, 0.64, 0.64, 0.64],
        [0.00, 0.00, 0.00, 0.00]], dtype=torch.float64)


## Problem 3: score centering

`q` is the engine's full distribution and `p` the trainer's. Score centering changes the gradient on the logits
from `A * (p - e_y)` to `A * (q - e_y)` (with weights of 1 and the engine's full distribution).

In [6]:
def score_centering_check():
    z = logits(W1, W2)                                  # (B, T, V)
    z.retain_grad()
    log_p = torch.log_softmax(z, -1)                    # (B, T, V)
    p = log_p.exp().detach()                            # (B, T, V)
    correction = ((q - p) * log_p).sum(-1)              # (B, T)   sum_k sg(q_k - p_k) * log p_k   (miles estimator.py:85-89)
    loss = (-A * (pick(log_p) - correction)).mean()     # ()       (miles estimator.py:90, importance weight "none")
    loss.backward()
    W1.grad, W2.grad = None, None
    A_logits = A.view(B, 1, 1)                          # (B, 1, 1)  the response's A, for every token and logit
    centered = A_logits * (q - e_y)                     # (B, T, V)
    plain = A_logits * (p - e_y)                        # (B, T, V)  REINFORCE
    return z.grad * N, centered, plain                  # each (B, T, V)

dz, centered, plain = score_centering_check()
print("autograd dz, token (0, 0)  :", dz[0, 0])
print("A * (q - e_y), token (0, 0):", centered[0, 0])
print("A * (p - e_y), REINFORCE   :", plain[0, 0])
print("all tokens match:", torch.allclose(dz, centered))

autograd dz, token (0, 0)  : tensor([-0.40,  0.01,  0.39], dtype=torch.float64)
A * (q - e_y), token (0, 0): tensor([-0.40,  0.01,  0.39], dtype=torch.float64)
A * (p - e_y), REINFORCE   : tensor([-0.10,  0.01,  0.09], dtype=torch.float64)
all tokens match: True


In [7]:
# Why: average the logits gradient over tokens the ENGINE samples (y ~ q).
#   REINFORCE: avg(A * (p - e_y)) = mean(A) * p - avg(A * e_y)
#   centered : avg(A * (q - e_y)) = mean(A) * q - avg(A * e_y)       avg(A * e_y) is the reward signal, same in both
# The difference, mean(A) * (p - q), is the extra push toward q that score centering removes. Below, A = +1 for every sample (mean(A) = 1, no reward signal),
# so only that extra term is left. With group-normalized advantages mean(A) is near 0 and the term is small.
with torch.no_grad():
    p00 = torch.softmax(logits(W1, W2), -1)[0, 0]                     # (V,)  trainer at position (0, 0)
    q00 = q[0, 0]                                                     # (V,)  engine at position (0, 0)
    E = torch.eye(V, **f64)                                           # (V, V) row k = one-hot of token k
    avg_plain = sum(q00[k] * (p00 - E[k]) for k in range(V))          # (V,)  = p - q
    avg_centered = sum(q00[k] * (q00 - E[k]) for k in range(V))       # (V,)  = 0
print("p (trainer):", p00, "  q (engine):", q00)
print("REINFORCE, average push      :", avg_plain)
print("score centering, average push:", avg_centered)
# miles records only the engine's top candidates per token (--rollout-top-logprobs-num, e.g. 128) and models the
# rest; this full-vocabulary version matches miles exactly only when the candidates cover the vocabulary.

p (trainer): tensor([0.90, 0.01, 0.09], dtype=torch.float64)   q (engine): tensor([0.60, 0.01, 0.39], dtype=torch.float64)
REINFORCE, average push      : tensor([ 0.30, -0.00, -0.30], dtype=torch.float64)
score centering, average push: tensor([-0.00, -0.00,  0.00], dtype=torch.float64)


## All per-token weights

**Table 2. `w` per method on this batch.** Response 0 has A = +1, response 1 has A = -1.

In [8]:
for name, fn in [("REINFORCE", reinforce), ("PPO", ppo), ("CISPO (paper)", cispo_paper), ("CISPO (miles)", cispo_miles),
                 ("TIS / decoupled PPO", tis), ("MIS", mis), ("IcePop", icepop), ("OPSM", opsm), ("GSPO", gspo)]:
    w = per_token_weight(fn).round(decimals=2)                         # (B, T)
    print(f"{name:20s} resp 0: {w[0].tolist()}   resp 1: {w[1].tolist()}")

# With one optimizer step per rollout, p_now = p_old, so r = s = 1: PPO, CISPO (paper), OPSM and GSPO change
# nothing. The engine mismatch (rho != 1) remains; TIS, MIS, IcePop, CISPO (miles) and score centering act on it.

REINFORCE            resp 0: [1.0, 1.0, 1.0, 1.0]   resp 1: [1.0, 1.0, 1.0, 1.0]
PPO                  resp 0: [1.05, 0.84, 0.38, 0.51]   resp 1: [0.0, 2.95, 0.0, 0.0]
CISPO (paper)        resp 0: [1.05, 0.84, 0.38, 0.51]   resp 1: [0.77, 2.95, 0.21, 0.74]
CISPO (miles)        resp 0: [1.51, 0.79, 0.29, 0.37]   resp 1: [1.07, 3.4, 0.06, 1.7]
TIS / decoupled PPO  resp 0: [1.51, 0.79, 0.29, 0.37]   resp 1: [0.0, 3.4, 0.0, 0.0]
MIS                  resp 0: [1.51, 0.79, 0.29, 0.37]   resp 1: [0.0, 3.4, 0.0, 0.0]
IcePop               resp 0: [1.51, 0.79, 0.29, 0.37]   resp 1: [0.0, 3.4, 0.0, 0.0]
OPSM                 resp 0: [1.05, 0.84, 0.38, 0.51]   resp 1: [0.0, 0.0, 0.0, 0.0]
GSPO                 resp 0: [0.64, 0.64, 0.64, 0.64]   resp 1: [0.0, 0.0, 0.0, 0.0]
